# Demo: generate a random OpenCellID gzip shard → `cell_towers/`

Each run reads the canonical dump **`310.csv.gz`** on the volume root, draws a **random sample**, and writes a **uniquely named** `*.csv.gz` directly under the UC volume folder **`cell_towers/`** (same path Auto Loader watches).

**Suggested demo sequence**

1. Run this notebook → confirm printed path under **`.../raw_data/cell_towers/`**.
2. **Catalog Explorer:** optional — verify the new file in **`cell_towers/`**.
3. Run **`network_analytics_pipeline`**.
4. **`SELECT COUNT(*) FROM ...bronze_cell_towers`**.
5. Repeat from step 1 for a second shard → pipeline → `COUNT(*)` again.

**Schedule:** With `continuous: false` on the pipeline, ingest runs when **you** trigger the pipeline; use a **scheduled Job** for hands-off repeats.

**Paths** (adjust if your bundle uses another catalog/schema/volume):  
- Source: `/Volumes/cmegdemos_catalog/network_analytics_enablement/raw_data/310.csv.gz`  
- Output: `/Volumes/cmegdemos_catalog/network_analytics_enablement/raw_data/cell_towers/`

In [ ]:
# Configuration
SOURCE_GZIP = "/Volumes/cmegdemos_catalog/network_analytics_enablement/raw_data/310.csv.gz"

# UC volume incoming folder for Auto Loader (bundle default: cell_towers_incoming_subdir)
CELL_TOWERS_DIR = "/Volumes/cmegdemos_catalog/network_analytics_enablement/raw_data/cell_towers"

COLUMNS = [
    "radio", "mcc", "net", "area", "cell", "unit",
    "lon", "lat", "range_m", "samples", "changeable",
    "created", "updated", "avg_signal",
]

MIN_SAMPLE_ROWS = 25_000
MAX_SAMPLE_ROWS = 120_000

In [ ]:
import os
import uuid
from datetime import datetime, timezone

import numpy as np
import pandas as pd

assert os.path.exists(SOURCE_GZIP), f"Missing source file: {SOURCE_GZIP}"
os.makedirs(CELL_TOWERS_DIR, exist_ok=True)

print(f"Reading {SOURCE_GZIP} ...")
df = pd.read_csv(
    SOURCE_GZIP,
    header=None,
    names=COLUMNS,
    compression="gzip",
    dtype=str,
    low_memory=False,
)
total = len(df)
print(f"Rows in source: {total:,}")

rng = np.random.default_rng()
hi = min(MAX_SAMPLE_ROWS, total)
lo = min(MIN_SAMPLE_ROWS, hi)
target_n = int(rng.integers(lo, hi + 1)) if hi >= lo else total
target_n = max(1, min(target_n, total))

sample = df.sample(n=target_n, replace=False, random_state=rng)
print(f"Sampled rows this run: {len(sample):,}")

ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
suffix = uuid.uuid4().hex[:10]
out_name = f"demo_shard_{ts}_{suffix}.csv.gz"
out_path = os.path.join(CELL_TOWERS_DIR, out_name)

sample.to_csv(out_path, header=False, index=False, compression="gzip")
print(f"\nWrote to cell_towers/:\n  {out_path}")
print("Next: run network_analytics_pipeline, then COUNT(*) on bronze_cell_towers.")

## After each pipeline run — bronze row count

Run in a SQL warehouse or in a notebook with Spark attached:

In [ ]:
try:
    spark.sql(
        "SELECT COUNT(*) AS bronze_cell_towers_rows FROM "
        "cmegdemos_catalog.network_analytics_enablement.bronze_cell_towers"
    ).show()
except NameError:
    print(
        "SELECT COUNT(*) AS bronze_cell_towers_rows FROM "
        "cmegdemos_catalog.network_analytics_enablement.bronze_cell_towers;"
    )